In [1]:
import numpy as np
import torch
import random
import shutil
import os
from pathlib import Path
from collections import defaultdict
from scipy.signal import hilbert

# ======================================================
# 🧭 Reproducibility
# ======================================================
random.seed(42)
np.random.seed(42)

# ======================================================
# 📂 Paths
# ======================================================
LOW_RES_DIR = Path("/home/g202393750/Documents/Superresolution/The definitive model/Data/Train/original/nx2.npy")
HIGH_RES_DIR = Path("/home/g202393750/Documents/Superresolution//The definitive model/Data/Train/original/sx.npy")
AUG_LOW_DIR = Path("/home/g202393750/Documents/Superresolution/The definitive model/Data/Train/data_aug_2/Aug_nx2.npy")
AUG_HIGH_DIR = Path("/home/g202393750/Documents/Superresolution/The definitive model/Data/Train/data_aug_2/Aug_sx.npy")

AUG_LOW_DIR.mkdir(parents=True, exist_ok=True)
AUG_HIGH_DIR.mkdir(parents=True, exist_ok=True)

# ======================================================
# ⚙️ Basic Augmentations
# ======================================================
def horizontal_flip(x): 
    return np.flip(x, axis=0)

def scale_amplitude(x): 
    return x * np.random.uniform(0.8, 1.2)

def trace_dropout(x, drop_prob=0.4):
    mask = np.random.rand(x.shape[0]) > drop_prob
    return x * mask[:, np.newaxis]

# ======================================================
# 🌋 Geophysically Realistic Noise Functions (Poststack)
# ======================================================
def coherent_lowfreq_noise(x, fs=500, fmin=5, fmax=15, amp=0.1):
    """Low-frequency coherent background noise (residual ground roll)."""
    n_traces, n_samples = x.shape
    t = np.arange(n_samples) / fs
    freq = np.random.uniform(fmin, fmax)
    phase = np.random.uniform(0, 2*np.pi)
    base = np.sin(2*np.pi*freq*t + phase)
    roll = np.tile(base, (n_traces, 1))
    roll *= (1 + 0.05*np.random.randn(n_traces, 1))
    return x + amp * roll

def smooth_random_noise(x, std=0.05, corr_len=5):
    """Smooth random noise — simulates mild random background."""
    n_traces, n_samples = x.shape
    noise = np.random.normal(0, std, size=x.shape)
    kernel = np.ones((corr_len, 1)) / corr_len
    smooth = np.apply_along_axis(lambda m: np.convolve(m, kernel.ravel(), mode='same'), axis=0, arr=noise)
    return x + smooth

def trace_gain_loss(x, prob=0.05, gain_min=0.5, gain_max=0.9):
    """Random trace amplitude dropouts (weak traces)."""
    mask = np.ones(x.shape[0])
    drop_idx = np.random.choice(np.arange(x.shape[0]), int(prob*x.shape[0]), replace=False)
    mask[drop_idx] = np.random.uniform(gain_min, gain_max, len(drop_idx))
    return x * mask[:, None]

def localized_spike(x, n_spikes=3, amp=0.4):
    """Random high-amplitude spikes (instrument or processing noise)."""
    n_traces, n_samples = x.shape
    x = x.copy()
    for _ in range(n_spikes):
        i = np.random.randint(0, n_traces)
        j = np.random.randint(0, n_samples)
        x[i, j] += amp * np.random.choice([-1, 1])
    return x

def mild_phase_shift(x, max_phase_deg=15):
    """Small phase rotation using Hilbert transform (residual migration/NMO)."""
    n_traces, n_samples = x.shape
    analytic = hilbert(x, axis=1)
    shift_deg = np.random.uniform(-max_phase_deg, max_phase_deg)
    shift_rad = np.deg2rad(shift_deg)
    rotated = np.real(analytic * np.exp(1j * shift_rad))
    return rotated

# ======================================================
# 🧩 Combine All Augmentations
# ======================================================
noise_funcs = [
    coherent_lowfreq_noise,
    smooth_random_noise,
    trace_gain_loss,
    localized_spike,
    mild_phase_shift
]

counter = defaultdict(int)
original_ids = sorted(os.listdir(LOW_RES_DIR))
new_id = 0

# ======================================================
# 📦 Copy Original Samples
# ======================================================
for file in original_ids:
    low = np.load(LOW_RES_DIR / file)
    high = np.load(HIGH_RES_DIR / file)
    np.save(AUG_LOW_DIR / f"{new_id:05}.npy", low)
    np.save(AUG_HIGH_DIR / f"{new_id:05}.npy", high)
    counter["Original"] += 1
    new_id += 1

# ======================================================
# 🔄 Augment Until Target Size (e.g., 10,000)
# ======================================================
TARGET_SIZE = 10000

while new_id < TARGET_SIZE:
    idx = random.randint(0, len(original_ids) - 1)
    low = np.load(LOW_RES_DIR / original_ids[idx])
    high = np.load(HIGH_RES_DIR / original_ids[idx])

    # Shared augmentations
    if random.random() < 0.5:
        low = horizontal_flip(low)
        high = horizontal_flip(high)
        counter["Horizontal Flip"] += 1

    if random.random() < 0.5:
        scale = np.random.uniform(0.8, 1.2)
        low *= scale
        high *= scale
        counter["Scale Amplitude"] += 1

    # Low-resolution specific augmentations
    if random.random() < 0.5:
        low = trace_dropout(low)
        counter["Trace Dropout"] += 1

    # Add geophysical noise with 70% chance
    if random.random() < 0.7:
        func = random.choice(noise_funcs)
        low = func(low)
        counter[func.__name__] += 1

    np.save(AUG_LOW_DIR / f"{new_id:05}.npy", low)
    np.save(AUG_HIGH_DIR / f"{new_id:05}.npy", high)
    new_id += 1

# ======================================================
# 📊 Final Report
# ======================================================
print("\n✅ Augmentation Summary:\n")
for key in [
    "Original", "Horizontal Flip", "Scale Amplitude", "Trace Dropout",
    "coherent_lowfreq_noise", "smooth_random_noise", 
    "trace_gain_loss", "localized_spike", "mild_phase_shift"
]:
    print(f"{key}: {counter[key]}")


✅ Augmentation Summary:

Original: 3200
Horizontal Flip: 3396
Scale Amplitude: 3374
Trace Dropout: 3417
coherent_lowfreq_noise: 971
smooth_random_noise: 953
trace_gain_loss: 939
localized_spike: 976
mild_phase_shift: 946


In [1]:
# import numpy as np
# import random
# import os
# from pathlib import Path
# from collections import defaultdict
# from scipy.signal import hilbert

# # ======================================================
# # 🔒 Reproducibility
# # ======================================================
# random.seed(42)
# np.random.seed(42)

# # ======================================================
# # 📂 Paths
# # ======================================================
# LOW_RES_DIR = Path("/home/g202393750/Documents/Superresolution/The definitive model/Data/Train/original/nx2.npy")
# HIGH_RES_DIR = Path("/home/g202393750/Documents/Superresolution/The definitive model/Data/Train/original/sx.npy")
# AUG_LOW_DIR = Path("/home/g202393750/Documents/Superresolution/The definitive model/Data/Train/data_aug/Aug_nx2.npy")
# AUG_HIGH_DIR = Path("/home/g202393750/Documents/Superresolution/The definitive model/Data/Train/data_aug/Aug_sx.npy")

# AUG_LOW_DIR.mkdir(parents=True, exist_ok=True)
# AUG_HIGH_DIR.mkdir(parents=True, exist_ok=True)

# # ======================================================
# # ⚙️ Basic Augmentations
# # ======================================================
# def horizontal_flip(x):
#     return np.flip(x, axis=0)

# def scale_amplitude(x):
#     return x * np.random.uniform(0.8, 1.2)

# def trace_dropout(x, drop_prob=0.4):
#     mask = np.random.rand(x.shape[0]) > drop_prob
#     return x * mask[:, np.newaxis]

# # ======================================================
# # 🌊 Geophysically Realistic Noise Functions (Poststack)
# # ======================================================
# def coherent_lowfreq_noise(x, fs=500, fmin=5, fmax=15, amp=0.1):
#     n_traces, n_samples = x.shape
#     t = np.arange(n_samples) / fs
#     freq = np.random.uniform(fmin, fmax)
#     phase = np.random.uniform(0, 2*np.pi)
#     base = np.sin(2*np.pi*freq*t + phase)
#     roll = np.tile(base, (n_traces, 1))
#     roll *= (1 + 0.05*np.random.randn(n_traces, 1))
#     return x + amp * roll

# def smooth_random_noise(x, std=0.05, corr_len=5):
#     n_traces, n_samples = x.shape
#     noise = np.random.normal(0, std, size=x.shape)
#     kernel = np.ones((corr_len, 1)) / corr_len
#     smooth = np.apply_along_axis(lambda m: np.convolve(m, kernel.ravel(), mode='same'), axis=0, arr=noise)
#     return x + smooth

# def trace_gain_loss(x, prob=0.05, gain_min=0.5, gain_max=0.9):
#     mask = np.ones(x.shape[0])
#     drop_idx = np.random.choice(np.arange(x.shape[0]), int(prob*x.shape[0]), replace=False)
#     mask[drop_idx] = np.random.uniform(gain_min, gain_max, len(drop_idx))
#     return x * mask[:, None]

# def localized_spike(x, n_spikes=3, amp=0.4):
#     n_traces, n_samples = x.shape
#     x = x.copy()
#     for _ in range(n_spikes):
#         i = np.random.randint(0, n_traces)
#         j = np.random.randint(0, n_samples)
#         x[i, j] += amp * np.random.choice([-1, 1])
#     return x

# def mild_phase_shift(x, max_phase_deg=15):
#     analytic = hilbert(x, axis=1)
#     shift_deg = np.random.uniform(-max_phase_deg, max_phase_deg)
#     shift_rad = np.deg2rad(shift_deg)
#     return np.real(analytic * np.exp(1j * shift_rad))

# noise_funcs = [
#     coherent_lowfreq_noise,
#     smooth_random_noise,
#     trace_gain_loss,
#     localized_spike,
#     mild_phase_shift
# ]

# # ======================================================
# # 📋 Step 1: Copy 3000 Original Samples
# # ======================================================
# counter = defaultdict(int)
# new_id = 0

# all_ids = sorted(os.listdir(LOW_RES_DIR))
# selected_ids = random.sample(all_ids, min(3000, len(all_ids)))


# for file in selected_ids:
#     low  = np.load(LOW_RES_DIR / file)
#     high = np.load(HIGH_RES_DIR / file)
#     np.save(AUG_LOW_DIR  / f"{new_id:05}.npy", low)
#     np.save(AUG_HIGH_DIR / f"{new_id:05}.npy", high)
#     counter["Original"] += 1
#     new_id += 1

# print(f"✅ Originals saved: {counter['Original']}")

# # ======================================================
# # 🔁 Step 2: Augment Until 10,000 Total
# # ======================================================
# TARGET_SIZE = 10000

# while new_id < TARGET_SIZE:
#     idx = random.randint(0, len(all_ids) - 1)
#     low  = np.load(LOW_RES_DIR / all_ids[idx])
#     high = np.load(HIGH_RES_DIR / all_ids[idx])

#     # Shared augmentations (applied to both low and high)
#     if random.random() < 0.5:
#         low  = horizontal_flip(low)
#         high = horizontal_flip(high)
#         counter["Horizontal Flip"] += 1

#     if random.random() < 0.5:
#         scale = np.random.uniform(0.8, 1.2)
#         low  *= scale
#         high *= scale
#         counter["Scale Amplitude"] += 1

#     # Low-resolution only augmentations
#     if random.random() < 0.5:
#         low = trace_dropout(low)
#         counter["Trace Dropout"] += 1

#     if random.random() < 0.7:
#         func = random.choice(noise_funcs)
#         low  = func(low)
#         counter[func.__name__] += 1

#     np.save(AUG_LOW_DIR  / f"{new_id:05}.npy", low)
#     np.save(AUG_HIGH_DIR / f"{new_id:05}.npy", high)
#     new_id += 1

# # ======================================================
# # 📊 Final Report
# # ======================================================
# print("\n✅ Augmentation Summary:\n")
# for key in [
#     "Original", "Horizontal Flip", "Scale Amplitude", "Trace Dropout",
#     "coherent_lowfreq_noise", "smooth_random_noise",
#     "trace_gain_loss", "localized_spike", "mild_phase_shift"
# ]:
#     print(f"  {key}: {counter[key]}")

# print(f"\n  Total samples saved: {new_id}")

✅ Originals saved: 3000

✅ Augmentation Summary:

  Original: 3000
  Horizontal Flip: 3485
  Scale Amplitude: 3467
  Trace Dropout: 3518
  coherent_lowfreq_noise: 1005
  smooth_random_noise: 977
  trace_gain_loss: 969
  localized_spike: 1013
  mild_phase_shift: 981

  Total samples saved: 10000
